## Import libraries


In [ ]:
import os
import pickle
import sys
import time

source_folder = "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src"
sys.path.append(source_folder)

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils.utils import evaluate_and_save_outputs, load_config, save_config, set_seed

plt.rcParams["font.family"] = "DeJavu Serif"
plt.rcParams["font.serif"] = "Times New Roman"

import warnings

warnings.filterwarnings("ignore")

out_dir = "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/output/figures"

## Read the Germany shapefile


In [ ]:
# Read the NUTS1 and NUTS3 shapefile for DE
de_nuts1_gdf = gpd.read_file(
    os.path.join("/beegfs/halder/DATA/", "DE_NUTS", "DE_NUTS_3.shp")
)
de_nuts1_gdf = de_nuts1_gdf[
    de_nuts1_gdf["LEVL_CODE"] == 1
]  # filter only NUT1 level code

de_nuts3_gdf = gpd.read_file(
    os.path.join("/beegfs/halder/DATA/", "DE_NUTS", "DE_NUTS_3.shp")
)
de_nuts3_gdf = de_nuts3_gdf[
    de_nuts3_gdf["LEVL_CODE"] == 3
]  # filter only NUT3 level code

fig, ax = plt.subplots(figsize=(8, 8))
de_nuts3_gdf.plot(
    ax=ax,
    column="NUTS_NAME",
    cmap="Set3",
    edgecolor="grey",
    linewidth=0.5,
    label="NUTS3",
)
de_nuts1_gdf.plot(ax=ax, facecolor="none", edgecolor="k", linewidth=1, label="NUTS1")
plt.show()

print(de_nuts1_gdf.shape, de_nuts3_gdf.shape)
de_nuts3_gdf.head()

## Prepare the results


In [ ]:
# Function to prepare results
def prepare_results(model_name, crop, month, baseline=False):
    target_mean = cfg.scalers["yield_mean"]
    target_std = cfg.scalers["yield_std"]

    train_dataset = pd.read_parquet(
        f"/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/data/processed/{crop}/timeseries/DE11A_2004.parquet"
    )
    train_dataset["date"] = pd.to_datetime(train_dataset["date"]).dt.strftime("%b-%d")
    date_list = train_dataset["date"].tolist()[: model_config.get("seq_length")]

    if baseline != True:
        result_dir = os.path.join(
            f"/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src/train/forecast/{crop}/{month}"
        )

    else:
        result_dir = os.path.join(
            f"/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src/train/baseline/{crop}/{month}/{model_name}"
        )

    with open(os.path.join(result_dir, f"train_outputs.pkl"), "rb") as f:
        train_results = pickle.load(f)

    with open(os.path.join(result_dir, f"validation_outputs.pkl"), "rb") as f:
        val_results = pickle.load(f)

    with open(os.path.join(result_dir, f"test_outputs.pkl"), "rb") as f:
        test_results = pickle.load(f)

    combined_results = {
        k: np.concatenate([train_results[k], test_results[k], val_results[k]], axis=0)
        for k in test_results.keys()
    }

    # Prepare final predictions
    preds = combined_results["prediction"] * target_std + target_mean
    actuals = combined_results["target"] * target_std + target_mean
    prediction_df = pd.DataFrame(
        preds, columns=[f"pred_yield_q{i}" for i in model_config["quantiles"]]
    )
    prediction_df["target_yield"] = actuals
    prediction_df["year"] = combined_results["year"]
    prediction_df["NUTS_ID"] = combined_results["NUTS_ID"]
    prediction_df = prediction_df[
        ["NUTS_ID", "year", "target_yield"]
        + [f"pred_yield_q{i}" for i in model_config["quantiles"]]
    ]
    prediction_df.sort_values(by=["year", "NUTS_ID"], inplace=True)
    prediction_df.reset_index(drop=True, inplace=True)

    if baseline == False:
        keys_to_keep = {
            "NUTS_ID",
            "year",
            "static_weights",
            "temporal_weights",
            "attention_weights",
        }

        weights = {
            k: combined_results[k] for k in keys_to_keep if k in combined_results
        }

        # Prepare the static weights dataframe
        static_weights_df = pd.DataFrame(
            weights["static_weights"][:, :, 0], columns=cfg.static_real_variables
        )
        static_weights_df["NUTS_ID"] = weights["NUTS_ID"]
        static_weights_df["year"] = weights["year"]
        static_weights_df = static_weights_df[
            ["NUTS_ID", "year"] + cfg.static_real_variables
        ]
        static_weights_df.sort_values(by=["year", "NUTS_ID"], inplace=True)
        static_weights_df.reset_index(drop=True, inplace=True)

        # Prepare the temporal weights dataframe
        temporal_weights_array = weights["temporal_weights"][:, :, :, 0]
        n_samples, n_timesteps, n_vars = temporal_weights_array.shape

        temporal_weights_df = pd.DataFrame(
            temporal_weights_array.reshape(-1, n_vars), columns=cfg.time_varying_real
        )

        temporal_weights_df["NUTS_ID"] = np.repeat(weights["NUTS_ID"], n_timesteps)
        temporal_weights_df["year"] = np.repeat(weights["year"], n_timesteps)
        temporal_weights_df["timestep"] = np.tile(range(len(date_list)), n_samples)

        temporal_weights_df = temporal_weights_df[
            ["NUTS_ID", "year", "timestep"] + cfg.time_varying_real
        ]

        temporal_weights_df.sort_values(
            by=["year", "NUTS_ID", "timestep"], inplace=True
        )

        temporal_weights_df.reset_index(drop=True, inplace=True)

        result = (
            prediction_df,
            static_weights_df,
            temporal_weights_df,
            date_list,
        )

        return result

    else:
        return (
            prediction_df,
            date_list,
        )

In [ ]:
# Get results for all crops for CropFusionNet and all baselines
crops = ["winter_wheat", "winter_barley", "silage_maize"]

crop_labels = {
    "winter_barley": "Winter Barley",
    "winter_wheat": "Winter Wheat",
    "silage_maize": "Silage Maize",
}
# Forecast month per crop
crop_months = {
    "winter_barley": "Jul",
    "winter_wheat": "Jul",
    "silage_maize": "Sep",
}

baseline_models = ["VanillaLSTM", "SimpleTransformer", "ResCNN"]
baseline_labels = {
    "VanillaLSTM": "Vanilla LSTM",
    "SimpleTransformer": "Simple Transformer",
    "ResCNN": "ResCNN",
}

# Load and store all results in a dictionary keyed by crop name
all_crop_results = {}
for crop_name in crops:
    global cfg, model_config, train_config
    cfg, model_config, train_config = load_config(crop_name)
    month = crop_months[crop_name]

    # CropFusionNet (main model)
    prediction_df, static_weights_df, temporal_weights_df, date_list = prepare_results(
        "CropFusionNet", crop=crop_name, month=month, baseline=False
    )
    all_crop_results[crop_name] = {
        "CropFusionNet": {
            "prediction_df": prediction_df,
            "static_weights_df": static_weights_df,
            "temporal_weights_df": temporal_weights_df,
            "date_list": date_list,
        }
    }
    print(f"[{crop_labels[crop_name]}] CropFusionNet: {prediction_df.shape[0]} samples")

    # Baseline models
    for model_name in baseline_models:
        pred_df_bl, date_list_bl = prepare_results(
            model_name, crop=crop_name, month=month, baseline=True
        )
        all_crop_results[crop_name][model_name] = {
            "prediction_df": pred_df_bl,
            "date_list": date_list_bl,
        }
        print(
            f"[{crop_labels[crop_name]}] {baseline_labels[model_name]}: {pred_df_bl.shape[0]} samples"
        )

## Plot the CropFusionNet prediction in timeseries


In [ ]:
label_map_local = {
    "target_yield": "Observed",
    "pred_yield_q0.5": "Predicted (Median)",
}
palette = {
    "Observed": "#2b8cbe",
    "Predicted (Median)": "#fd8d3c",
}

panel_labels = ["(a)", "(b)", "(c)"]


fig, axes = plt.subplots(
    len(crops), 1, figsize=(12, 3 * len(crops)), dpi=300, sharex=True
)

for i, (ax, crop_name) in enumerate(zip(axes, crops)):
    pred_df = all_crop_results[crop_name]["CropFusionNet"]["prediction_df"].copy()

    # Melt to long format
    pred_df = pred_df.melt(
        id_vars=["year"],
        value_vars=["target_yield", "pred_yield_q0.5"],
        var_name="type",
        value_name="yield",
    )
    pred_df["year"] = pred_df["year"].astype(int)
    pred_df["type"] = pred_df["type"].map(label_map_local)

    sns.boxplot(
        data=pred_df,
        x="year",
        y="yield",
        hue="type",
        palette=palette,
        linewidth=0.75,
        ax=ax,
        legend=True,
        flierprops=dict(marker="o", markersize=3, markerfacecolor="gray", alpha=0.6),
    )

    ax.text(
        0.01,
        0.95,
        panel_labels[i],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
    )

    mean_obs = pred_df.loc[pred_df["type"] == "Observed", "yield"].mean()
    mean_pred = pred_df.loc[pred_df["type"] == "Predicted (Median)", "yield"].mean()

    ax.axhline(
        mean_obs,
        color=palette["Observed"],
        linestyle="--",
        linewidth=1,
        label="Observed mean",
    )

    ax.axhline(
        mean_pred,
        color=palette["Predicted (Median)"],
        linestyle="--",
        linewidth=1,
        label="Predicted mean",
    )

    years_sorted = sorted(pred_df["year"].unique())
    year_to_pos_local = {year: i for i, year in enumerate(years_sorted)}

    # Shade train / validation / test regions
    ax.axvspan(
        -0.5,
        year_to_pos_local[2018] + 0.5,
        color="#caffbf",
        alpha=0.3,
        label="Training",
    )
    ax.axvspan(
        year_to_pos_local[2019] - 0.5,
        year_to_pos_local[2021] + 0.5,
        color="#a0c4ff",
        alpha=0.3,
        label="Validation",
    )
    ax.axvspan(
        year_to_pos_local[2022] - 0.5,
        len(years_sorted) - 0.5,
        color="#ffadad",
        alpha=0.3,
        label="Testing",
    )

    ax.set_xlim(-0.5, len(years_sorted) - 0.5)

    ax.tick_params(axis="x", rotation=90, labelsize=11)
    ax.tick_params(axis="y", labelsize=11)
    ax.set_xlabel("\nYear", fontsize=12)
    ax.set_ylabel("Yield ($t \\ ha^{-1}$)", fontsize=12)


# Create a single legend for the figure
handles, labels = axes[-1].get_legend_handles_labels()  # get from last axis
unique_handles_labels = dict(zip(labels, handles))  # remove duplicates

fig.legend(
    unique_handles_labels.values(),
    unique_handles_labels.keys(),
    fontsize=10,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
    ncol=7,
)

for ax in axes:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

plt.tight_layout(h_pad=1)
# plt.savefig(
#     os.path.join(out_dir, "final", "obs_vs_predicted_yield_distribution_per_year.svg"),
#     bbox_inches="tight",
#     dpi=300,
# )
# plt.savefig(
#     os.path.join(out_dir, "final", "obs_vs_predicted_yield_distribution_per_year.png"),
#     bbox_inches="tight",
#     dpi=300,
# )
plt.show()

## Plot the Comparison Scatter Plot


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, pearsonr
from sklearn.metrics import r2_score, mean_squared_error
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

# All models in display order: CropFusionNet first, then baselines
all_models = ["CropFusionNet"] + baseline_models
model_display_labels = {
    "CropFusionNet": "CropFusionNet",
    **baseline_labels,
}

n_crops = len(crops)
n_models = len(all_models)
panel_alpha = [chr(ord("a") + i) for i in range(n_crops * n_models)]

fig, axes = plt.subplots(
    n_crops, n_models, figsize=(3 * n_models, 3 * n_crops), dpi=300
)

# Ensure axes is a 2D array even if there's only 1 crop or 1 model
if n_crops == 1 and n_models == 1:
    axes = np.array([[axes]])
elif n_crops == 1:
    axes = axes[np.newaxis, :]
elif n_models == 1:
    axes = axes[:, np.newaxis]

row_scatters = {}  # store the last scatter per row for colorbar

for row, crop_name in enumerate(crops):
    for col, model_name in enumerate(all_models):
        ax = axes[row, col]
        panel_idx = row * n_models + col

        pred_df = all_crop_results[crop_name][model_name]["prediction_df"].copy()
        pred_df["year"] = pred_df["year"].astype(int)

        # Use validation + test years only
        sel = pred_df[pred_df["year"] >= 2019]

        yhat = sel["pred_yield_q0.5"].values
        ytrue = sel["target_yield"].values

        # KDE-based point density for coloring
        xy = np.vstack([ytrue, yhat])
        density = gaussian_kde(xy)(xy)
        idx = density.argsort()
        x_plot, y_plot, d_plot = ytrue[idx], yhat[idx], density[idx]

        sc = ax.scatter(
            x_plot,
            y_plot,
            c=d_plot,
            s=10,
            cmap="viridis",
            edgecolors="none",
            alpha=0.7,
        )
        row_scatters[row] = sc  # keep last scatter for colorbar

        # 1:1 line & axis limits
        vmin = min(ytrue.min(), yhat.min())
        vmax = max(ytrue.max(), yhat.max())

        # Add buffer to limits so points aren't right on the edge
        buffer = (vmax - vmin) * 0.05
        ax.set_xlim(vmin - buffer, vmax + buffer)
        ax.set_ylim(vmin - buffer, vmax + buffer)

        # Force the subplot to be a square
        ax.set_aspect("equal", adjustable="box")

        ax.plot([vmin, vmax], [vmin, vmax], color="red", lw=1, label="1:1")

        # Trend line
        slope, intercept = np.polyfit(ytrue, yhat, 1)
        xline = np.linspace(vmin, vmax, 100)
        ax.plot(xline, slope * xline + intercept, "k--", lw=1, label="Trend")

        # Metrics
        r2 = r2_score(ytrue, yhat)
        rmse = math.sqrt(mean_squared_error(ytrue, yhat))
        r, _ = pearsonr(ytrue, yhat)

        ax.text(
            0.04,
            0.96,
            f"r = {r:.3f}\nR² = {r2:.3f}\nRMSE = {rmse:.2f}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=11,
            bbox=dict(
                boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="none"
            ),
        )

        # Panel label
        ax.text(
            0.97,
            0.04,
            f"({panel_alpha[panel_idx]})",
            transform=ax.transAxes,
            va="bottom",
            ha="right",
            fontsize=14,
            fontweight="bold",
        )

        # Titles and axis labels only on borders
        if row == 0:
            ax.set_title(
                model_display_labels[model_name], fontsize=11, fontweight="bold"
            )
        if col == 0:
            ax.set_ylabel(
                f"{crop_labels[crop_name]}\n\nPredicted ($t\\ ha^{{-1}}$)", fontsize=12
            )
        else:
            ax.set_ylabel("")

        if row == n_crops - 1:
            ax.set_xlabel("Observed ($t\\ ha^{-1}$)", fontsize=12)
        else:
            ax.set_xlabel("")

        ax.tick_params(labelsize=10)

# Shared legend at the bottom
legend_handles = [
    Line2D([0], [0], color="red", lw=1, label="1:1 line"),
    Line2D([0], [0], color="black", lw=1, linestyle="--", label="Trend line"),
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.05),
    ncol=2,
    fontsize=14,
    frameon=False,
)

plt.tight_layout(h_pad=1.5, w_pad=1)

fig.canvas.draw()
for row in range(n_crops):
    last_ax = axes[row, -1]
    pos = last_ax.get_position()
    cbar_width = 0.012
    cbar_pad = 0.008
    cax = fig.add_axes([pos.x1 + cbar_pad, pos.y0, cbar_width, pos.height])
    cbar = fig.colorbar(row_scatters[row], cax=cax)
    cbar.set_label("Density", fontsize=10)
    cbar.ax.tick_params(labelsize=8)

# plt.savefig(
#     os.path.join(out_dir, "final", "scatter_all_crops_all_models.svg"),
#     bbox_inches="tight",
#     dpi=300,
# )
# plt.savefig(
#     os.path.join(out_dir, "final", "scatter_all_crops_all_models.png"),
#     bbox_inches="tight",
#     dpi=300,
# )
plt.show()

## Metrics Table — All Crops × All Models


In [ ]:
import math
from scipy.stats import pearsonr
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
)

all_models_table = ["CropFusionNet"] + baseline_models

rows = []
for crop_name in crops:
    for model_name in all_models_table:
        pred_df = all_crop_results[crop_name][model_name]["prediction_df"].copy()
        pred_df["year"] = pred_df["year"].astype(int)

        for split_label, condition in [
            ("Validation", (pred_df["year"] >= 2019) & (pred_df["year"] <= 2021)),
            ("Test", (pred_df["year"] >= 2022)),
            ("Val + Test", (pred_df["year"] >= 2019)),
        ]:
            sel = pred_df[condition]
            ytrue = sel["target_yield"].values
            yhat = sel["pred_yield_q0.5"].values

            r2 = r2_score(ytrue, yhat)
            rmse = math.sqrt(mean_squared_error(ytrue, yhat))
            mae = mean_absolute_error(ytrue, yhat)
            mape = mean_absolute_percentage_error(ytrue, yhat) * 100  # in %
            r, _ = pearsonr(ytrue, yhat)
            bias = float(np.mean(yhat - ytrue))

            rows.append(
                {
                    "Crop": crop_labels[crop_name],
                    "Model": model_display_labels[model_name],
                    "Split": split_label,
                    "r": round(r, 3),
                    "R²": round(r2, 3),
                    "RMSE": round(rmse, 3),
                    "MAE": round(mae, 3),
                    "MAPE (%)": round(mape, 2),
                    "Bias": round(bias, 3),
                }
            )

metrics_df = pd.DataFrame(rows)

# Multi-index display: Crop → Model → Split
metrics_df_display = metrics_df.set_index(["Crop", "Model", "Split"])

display(
    metrics_df_display.style.format(precision=3)
    .set_caption(
        "Predictive performance metrics — all crops × all models × data splits"
    )
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("font-size", "13px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                ],
            },
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "center")]},
        ]
    )
    .background_gradient(subset=["R²", "r"], cmap="RdYlGn", axis=0)
    .background_gradient(subset=["RMSE", "MAE", "MAPE (%)"], cmap="RdYlGn_r", axis=0)
)

In [ ]:
# Save the table
# metrics_df[metrics_df["Split"] == "Val + Test"].to_csv(
#     os.path.join(
#         "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/output/",
#         "tables",
#         "validation_metrics.csv",
#     ),
#     index=False,
# )

## Bar Plot — NRMSE & MAPE across All Models and Crops


In [ ]:
# Use the Val + Test split for the bar plot
plot_df = metrics_df[metrics_df["Split"] == "Val + Test"].copy()

# Compute NRMSE = RMSE / mean(observed) * 100  (%) per crop × model
nrmse_rows = []
for crop_name in crops:
    for model_name in ["CropFusionNet"] + baseline_models:
        pred_df = all_crop_results[crop_name][model_name]["prediction_df"].copy()
        pred_df["year"] = pred_df["year"].astype(int)
        sel = pred_df[pred_df["year"] >= 2019]
        ytrue = sel["target_yield"].values
        yhat = sel["pred_yield_q0.5"].values
        rmse_val = math.sqrt(mean_squared_error(ytrue, yhat))
        nrmse_val = rmse_val / ytrue.mean() * 100
        nrmse_rows.append(
            {
                "Crop": crop_labels[crop_name],
                "Model": model_display_labels[model_name],
                "NRMSE (%)": round(nrmse_val, 2),
            }
        )

nrmse_df = pd.DataFrame(nrmse_rows)
plot_df = plot_df.merge(nrmse_df, on=["Crop", "Model"])

# Pre-compute the best (minimum) value per crop for each metric
best_vals = {
    metric: plot_df.groupby("Crop")[metric].min().to_dict()
    for metric in ["NRMSE (%)", "MAPE (%)"]
}

# Ordered model list and colour palette
model_order = ["CropFusionNet", "Vanilla LSTM", "Simple Transformer", "ResCNN"]
crop_order = [crop_labels[c] for c in crops]

model_colors = {
    "CropFusionNet": "#688ae8",
    "Vanilla LSTM": "#da7596",
    "Simple Transformer": "#2ea597",
    "ResCNN": "#e07941",
}

metrics_to_plot = ["NRMSE (%)", "MAPE (%)"]
metric_ylabels = {"NRMSE (%)": "NRMSE (%)", "MAPE (%)": "MAPE (%)"}
panel_labels_bar = ["(a)", "(b)"]

n_metrics = len(metrics_to_plot)
n_crops_ = len(crop_order)
n_models_ = len(model_order)

bar_width = 0.18
group_gap = 0.1
x_positions = np.arange(n_crops_) * (n_models_ * bar_width + group_gap)

fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4), dpi=300)

for ax_idx, (ax, metric) in enumerate(zip(axes, metrics_to_plot)):
    for i, model_name in enumerate(model_order):
        model_data = plot_df[plot_df["Model"] == model_name]
        vals = [
            model_data.loc[model_data["Crop"] == c, metric].values[0]
            for c in crop_order
        ]
        offset = (i - (n_models_ - 1) / 2) * bar_width
        bars = ax.bar(
            x_positions + offset,
            vals,
            width=bar_width,
            color=model_colors[model_name],
            label=model_name,
            edgecolor="k",
            linewidth=0.5,
            alpha=0.8,
        )
        # Value labels — bold + larger font if this bar is the best for its crop
        for bar, val, crop in zip(bars, vals, crop_order):
            is_best = val == best_vals[metric][crop]
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.02,
                f"{val:.1f}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold" if is_best else "normal",
                rotation=90,
                color="black",
            )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(crop_order, fontsize=11)
    ax.set_ylabel(metric_ylabels[metric], fontsize=11)
    ax.tick_params(axis="y", labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
    ax.set_axisbelow(True)

    # Panel label (a) / (b)
    ax.text(
        0.02,
        0.98,
        panel_labels_bar[ax_idx],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
    )

# Re-apply y-limits after drawing to give room for value labels
for ax in axes:
    ax.set_ylim(0, ax.get_ylim()[1] * 1.2)

# Single shared legend
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=model_colors[m], label=m) for m in model_order
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.06),
    ncol=n_models_,
    fontsize=10,
    frameon=False,
)

plt.tight_layout()
# plt.savefig(
#     os.path.join(out_dir, "final", "barplot_nrmse_mape_all_crops_models.svg"),
#     bbox_inches="tight",
#     dpi=300,
# )
# plt.savefig(
#     os.path.join(out_dir, "final", "barplot_nrmse_mape_all_crops_models.png"),
#     bbox_inches="tight",
#     dpi=300,
# )
plt.show()

## Country-level relative error (%) per year for CropFusionNet


In [ ]:
#  RE_year = (mean_pred_year - mean_obs_year) / mean_obs_year * 100

re_rows = []
for crop_name in crops:
    pred_df = all_crop_results[crop_name]["CropFusionNet"]["prediction_df"].copy()
    pred_df["year"] = pred_df["year"].astype(int)

    yearly = (
        pred_df.groupby("year")[["target_yield", "pred_yield_q0.5"]]
        .mean()
        .reset_index()
    )
    yearly["RE (%)"] = (
        (yearly["pred_yield_q0.5"] - yearly["target_yield"])
        / yearly["target_yield"]
        * 100
    )
    yearly["Crop"] = crop_labels[crop_name]
    re_rows.append(
        yearly[["Crop", "year", "target_yield", "pred_yield_q0.5", "RE (%)"]]
    )

re_df = pd.concat(re_rows, ignore_index=True)

# ---- Plot ----
n_c = len(crops)
panel_re = ["(a)", "(b)", "(c)"]

fig, axes = plt.subplots(n_c, 1, figsize=(11, 3.2 * n_c), dpi=300, sharex=False)

for i, (ax, crop_name) in enumerate(zip(axes, crops)):
    sub = re_df[re_df["Crop"] == crop_labels[crop_name]].sort_values("year")
    years = sub["year"].values
    re = sub["RE (%)"].values

    # Colour bars by sign: over- vs under-prediction
    colors = ["#e07941" if v > 0 else "#688ae8" for v in re]
    bars = ax.bar(
        years, re, color=colors, edgecolor="k", linewidth=0.4, width=0.7, zorder=3
    )

    # Zero line
    ax.axhline(0, color="black", linewidth=0.8, zorder=4)

    # Shade validation / test bands
    val_years = [y for y in years if 2019 <= y <= 2021]
    test_years = [y for y in years if y >= 2022]
    if val_years:
        ax.axvspan(
            min(val_years) - 0.5,
            max(val_years) + 0.5,
            color="#a0c4ff",
            alpha=0.25,
            zorder=0,
            label="Validation",
        )
    if test_years:
        ax.axvspan(
            min(test_years) - 0.5,
            max(test_years) + 0.5,
            color="#ffadad",
            alpha=0.25,
            zorder=0,
            label="Test",
        )

    # Annotate each bar
    for bar, v in zip(bars, re):
        va = "bottom" if v >= 0 else "top"
        ypos = v + 0.3 if v >= 0 else v - 0.3
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            ypos,
            f"{v:+.1f}%",
            ha="center",
            va=va,
            fontsize=7.5,
            zorder=5,
        )

    ax.set_xlim(years[0] - 0.6, years[-1] + 0.6)
    ax.set_xticks(years)
    ax.set_xticklabels(years, fontsize=9)
    ax.tick_params(axis="y", labelsize=9)
    ax.set_ylabel("Relative Error (%)", fontsize=10)
    ax.set_title(crop_labels[crop_name], fontsize=11, fontweight="bold")
    ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Panel label
    ax.text(
        0.01,
        0.97,
        panel_re[i],
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        va="top",
        ha="left",
    )

# Shared legend
from matplotlib.patches import Patch

legend_items = [
    Patch(color="#e07941", label="Over-prediction"),
    Patch(color="#688ae8", label="Under-prediction"),
    Patch(color="#a0c4ff", alpha=0.5, label="Validation period"),
    Patch(color="#ffadad", alpha=0.5, label="Test period"),
]
fig.legend(
    handles=legend_items,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
    ncol=4,
    fontsize=10,
    frameon=False,
)

plt.tight_layout(h_pad=1.5)
# plt.savefig(os.path.join(out_dir, "final", "country_level_relative_error_per_year.svg"), bbox_inches="tight", dpi=300)
# plt.savefig(os.path.join(out_dir, "final", "country_level_relative_error_per_year.png"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Save the table
# re_df.to_csv(
#     os.path.join(
#         "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/output/",
#         "tables",
#         "country_level_relative_error_per_year.csv",
#     ),
#     index=False,
# )